# LSTM Multi-City Dataset Construction

This notebook builds the H3-aggregated demand dataset used by the LSTM model.
Starting from the cleaned trip records (10 European cities), it assigns each trip
to an H3 hexagonal cell (resolution 8), aggregates pickup counts by (city, cell,
date, hour), expands the grid with zero-demand slots, and engineers temporal features.

The output is a single Parquet file consumed by the LSTM training notebook.

## 1. Configuration and Imports

In [ ]:
import pandas as pd
import numpy as np
import h3
from pathlib import Path
from itertools import product
import warnings
warnings.filterwarnings('ignore')

from trip_cleaning import parse_date_utc

CLEANED_PATH = Path('../results/trips_cleaned.parquet')
OUTPUT_DIR   = Path('../results/lstm')

H3_RESOLUTION = 7

CITIES = [
    'amsterdam', 'berlin', 'firenze', 'kobenhavn',
    'milano', 'muenchen', 'roma', 'stockholm', 'torino', 'wien'
]

print(f'H3 resolution: {H3_RESOLUTION}')
print(f'Cities to process: {len(CITIES)}')

## 2. H3 Spatial Assignment and Temporal Parsing

Each trip is assigned to an H3 hexagonal cell based on its origin coordinates.
Timestamps are converted to UTC so that all cities share a common absolute time
reference.

In [ ]:
def coord_to_h3(lat, lon, resolution=H3_RESOLUTION):
    return h3.latlng_to_cell(lat, lon, resolution)


def process_city(df_city, city, resolution=H3_RESOLUTION):
    df = df_city.copy()
    df["datetime_utc"] = df["s_date"].apply(parse_date_utc)
    df = df.dropna(subset=["datetime_utc"])
    df["h3_cell"] = df.apply(lambda r: coord_to_h3(r["lat"], r["lon"]), axis=1)
    df["date"]        = df["datetime_utc"].dt.date
    df["hour"]        = df["datetime_utc"].dt.hour
    df["day_of_week"] = df["datetime_utc"].dt.dayofweek
    df["month"]       = df["datetime_utc"].dt.month
    df["city"]        = city
    return df[["city", "h3_cell", "datetime_utc", "date",
               "hour", "day_of_week", "month"]]

In [ ]:
df_cleaned = pd.read_parquet(CLEANED_PATH)
print(f"Cleaned dataset: {len(df_cleaned):,} trips, {df_cleaned['city'].nunique()} cities")

all_trips = []
for city in CITIES:
    print(f'\n[{city}] H3 resolution: {H3_RESOLUTION}')
    df_city = df_cleaned[df_cleaned['city'] == city]
    df_processed = process_city(df_city, city)
    print(f'  Processed trips: {len(df_processed):,}')
    all_trips.append(df_processed)

df_trips = pd.concat(all_trips, ignore_index=True)
print(f'\nTotal trips: {len(df_trips):,}')

## 3. Demand Aggregation

Trip counts are grouped by (city, H3 cell, date, hour) to obtain the hourly
pickup demand per hexagonal cell.

In [ ]:
df_demand = (
    df_trips
    .groupby(['city', 'h3_cell', 'date', 'hour', 'day_of_week', 'month'])
    .size()
    .reset_index(name='target_demanda')
)

print(f'Rows with demand > 0: {len(df_demand):,}')
print(f'Mean demand per cell-hour: {df_demand["target_demanda"].mean():.2f}')

celdas_por_ciudad = df_demand.groupby('city')['h3_cell'].nunique().reset_index()
celdas_por_ciudad.columns = ['city', 'num_cells']
print(celdas_por_ciudad.sort_values('num_cells', ascending=False).to_string(index=False))

## 4. Zero Expansion (Cartesian Product)

For each city, a full Cartesian product of (active H3 cells x dates x hours) is
constructed. Slots without observed trips are filled with zero demand. This
explicit representation of zeros is essential for the LSTM to learn the baseline
absence of demand.

In [ ]:
city_frames = []

for city in CITIES:
    df_c = df_demand[df_demand['city'] == city].copy()
    if df_c.empty:
        continue

    active_cells = df_c['h3_cell'].unique()
    all_dates    = pd.date_range(df_c['date'].min(), df_c['date'].max(), freq='D').date
    all_hours    = range(24)

    full_index = pd.DataFrame(
        list(product(active_cells, all_dates, all_hours)),
        columns=['h3_cell', 'date', 'hour']
    )
    full_index['city'] = city

    df_full = full_index.merge(
        df_c[['city', 'h3_cell', 'date', 'hour', 'day_of_week', 'month', 'target_demanda']],
        on=['city', 'h3_cell', 'date', 'hour'], how='left'
    )
    df_full['target_demanda'] = df_full['target_demanda'].fillna(0).astype(np.int16)

    df_full['date_dt']      = pd.to_datetime(df_full['date'])
    df_full['day_of_week']  = df_full['date_dt'].dt.dayofweek
    df_full['month']        = df_full['date_dt'].dt.month
    df_full.drop(columns='date_dt', inplace=True)

    city_frames.append(df_full)
    pct_zero = (df_full['target_demanda'] == 0).mean() * 100
    print(f'{city:12s} | {len(df_full):8,} rows | active cells: {len(active_cells):5,} | zeros: {pct_zero:.1f}%')

dataset = pd.concat(city_frames, ignore_index=True)
print(f'\nTotal rows: {len(dataset):,}')

## 5. Cyclic Temporal Features

Hour and day-of-week are encoded as sine/cosine pairs to preserve their circular
nature (e.g., hour 23 is close to hour 0). A binary weekend flag is also added.

In [ ]:
dataset['hour_sin'] = np.sin(2 * np.pi * dataset['hour'] / 24)
dataset['hour_cos'] = np.cos(2 * np.pi * dataset['hour'] / 24)
dataset['dow_sin']  = np.sin(2 * np.pi * dataset['day_of_week'] / 7)
dataset['dow_cos']  = np.cos(2 * np.pi * dataset['day_of_week'] / 7)
dataset['is_weekend'] = (dataset['day_of_week'] >= 5).astype(np.int8)

print('Cyclic features added.')

## 6. Lag Features

Autoregressive lag features capture recent demand history. Lags at 1 hour, 24 hours,
and 168 hours (one week) are computed per (city, H3 cell) group, along with a
3-hour rolling mean.

In [ ]:
dataset.sort_values(['city', 'h3_cell', 'date', 'hour'], inplace=True)
dataset.reset_index(drop=True, inplace=True)

g = dataset.groupby(['city', 'h3_cell'])['target_demanda']

dataset['lag_1h']          = g.shift(1)
dataset['lag_24h']         = g.shift(24)
dataset['lag_168h']        = g.shift(168)
dataset['rolling_mean_3h'] = g.transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)

lag_cols = ['lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_3h']
print('NaN per lag column:')
print(dataset[lag_cols].isna().sum())

## 7. Summary Statistics

In [ ]:
summary = dataset.groupby('city').agg(
    rows           = ('target_demanda', 'count'),
    h3_cells       = ('h3_cell', 'nunique'),
    mean_demand    = ('target_demanda', 'mean'),
    max_demand     = ('target_demanda', 'max'),
    pct_zeros      = ('target_demanda', lambda x: (x == 0).mean() * 100),
    date_start     = ('date', 'min'),
    date_end       = ('date', 'max'),
).round(2)

print(summary.to_string())

## 8. Saving the Dataset

In [ ]:
COLS_FINAL = [
    'city', 'h3_cell', 'date', 'hour',
    'day_of_week', 'month', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'target_demanda',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_3h'
]

dataset_final = dataset[COLS_FINAL].copy()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / 'dataset_h3_multicidad.parquet'
dataset_final.to_parquet(output_path, index=False)

print(f'Saved: {output_path}')
print(f'Size: {output_path.stat().st_size / 1024**2:.1f} MB')
print(f'Shape: {dataset_final.shape}')